# CLIP features
Turns every face crop into 512 numbers with a frozen CLIP image encoder (ViT-B/16, OpenAI weights).

In [1]:
config = {
    "limit": 0,
    # the OpenAI weights were trained with QuickGELU
    "model": "ViT-B-16-quickgelu",
    "pretrained": "openai",
    "batch_size": 256,
    "workers": 8,
    "out": "/kaggle/working/features",
}
parts = ["ffpp/train", "ffpp/val", "ffpp/test", "celebdf/test"]

In [2]:
!pip install -q open_clip_torch==3.3.0

In [3]:
import os
import glob

import numpy as np
import pandas as pd
import torch
from PIL import Image
import open_clip

In [4]:
def find(pattern):
    for depth in range(8):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + pattern)
        if len(hits) > 0:
            return hits[0]
    raise FileNotFoundError(pattern)


class Crops(torch.utils.data.Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.preprocess(Image.open(self.paths[i]).convert("RGB"))

In [5]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
model, _, preprocess = open_clip.create_model_and_transforms(config["model"], pretrained=config["pretrained"])
model = model.to(device).eval()

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

In [6]:
os.makedirs(config["out"], exist_ok=True)
for part in parts:
    name = part.replace("/", "_")
    rows = pd.read_csv(find(name + "_manifest.csv"), dtype={"video_id": str})
    root = find(part)[:-len(part)]
    if config["limit"] > 0:
        videos = rows[["method", "video_id"]].drop_duplicates().head(config["limit"])
        rows = rows.merge(videos)
    paths = [os.path.join(root, p) for p in rows["path"]]
    missing = [p for p in paths if not os.path.exists(p)]
    assert len(missing) == 0, missing[:3]

    loader = torch.utils.data.DataLoader(Crops(paths, preprocess), batch_size=config["batch_size"],
                                         num_workers=config["workers"])
    chunks = []
    with torch.no_grad(), torch.autocast("cuda"):
        for batch in loader:
            chunks.append(model.encode_image(batch.to(device)).float().cpu().numpy())
    features = np.concatenate(chunks)

    np.save(os.path.join(config["out"], name + "_clip.npy"), features)
    rows.to_csv(os.path.join(config["out"], name + "_rows.csv"), index=False)
    print(name, features.shape, "mean norm %.2f" % np.linalg.norm(features, axis=1).mean())

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


ffpp_train (46076, 512) mean norm 10.49


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


ffpp_val (22396, 512) mean norm 10.48


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


ffpp_test (22400, 512) mean norm 10.55


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


celebdf_test (16572, 512) mean norm 10.81
